# Part 2 — train T5-small and back up to Drive

Use the **same Colab GPU runtime** where Section 9 printed `Q5 EVALUATION CHECK PASSED`. This notebook does not edit homework files. It snapshots your checked code, then calls your `train_epoch` and `eval_epoch` functions from an operational loop that adds backups and early stopping. It deliberately does not call the starter `main()` or load the test dataset.

Run **Preparation**, then run **Train** once. Do not use Run All or start another training process. No additional homework-code edits are needed.

Initial development experiment: pretrained `google-t5/t5-small`, all parameters trainable; AdamW, learning rate 3e-4, weight decay 0, batch size 8, seed 42, FP32, linear learning-rate schedule with one warmup epoch; at most 20 epochs, early stopping after four epochs without strictly improved dev Record F1. Greedy decoding uses at most 512 new tokens. Keep at least about **1 GB free in Drive**, in addition to previous assignment backups.

Each completed training epoch is saved to `checkpoints/last.tar` in Drive **before dev evaluation**. The best dev model is kept in `checkpoints/best.tar`; a new archive replaces the previous one only after creation and basic validation. Logs, settings, source/data snapshot, per-epoch metrics, best/last dev predictions, and a final report are also saved under `MyDrive/NYU_NLP_HW4/part2/training_runs/<timestamp>/`.

**Limits:** checkpoints contain model weights/config/tokenizer, not optimizer/scheduler/RNG state, so they support inference but not exact training resumption. If Colab destroys the runtime mid-epoch, unfinished work since the last completed checkpoint is lost. Before the first epoch completes, no trained-model backup exists. These backups do not prevent Colab disconnections or runtime limits: https://research.google.com/colaboratory/faq.html. Leave the computer powered and connected; the notebook does not use keep-alive workarounds.

After training, save this notebook and ask for review before generating the held-out test submission. Actual dev results, not diagnostic fixture scores, will populate the report.


## 1. Preparation

Run this first and complete Drive authorization. It verifies that your uploaded code matches the successful Section 9 snapshot. Preparation does not train.


In [1]:
RUNNER_SOURCE = "\"\"\"Operational Colab runner: use homework functions without editing homework files.\n\nOnly train/dev are loaded. Checkpoint archives contain weights/config/tokenizer,\nnot optimizer state; they support later inference, not exact training resumption.\n\"\"\"\n\nimport gc\nimport hashlib\nimport importlib.metadata\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport shutil\nimport sys\nimport tarfile\nimport time\nfrom datetime import datetime, timezone\nfrom types import SimpleNamespace\n\n\ndef write_json(path, value):\n    path = Path(path)\n    temporary = path.with_name(path.name + \".partial\")\n    temporary.write_text(json.dumps(value, indent=2))\n    os.replace(temporary, path)\n\n\ndef publish_checkpoint(source, destination):\n    \"\"\"Finish a new archive before replacing this run's previous archive.\"\"\"\n    source, destination = Path(source), Path(destination)\n    required = {\"config.json\", \"model.safetensors\", \"tokenizer.json\", \"checkpoint_meta.json\"}\n    assert all((source / name).is_file() for name in required)\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    temporary = destination.with_name(destination.name + \".partial\")\n    with tarfile.open(temporary, \"w\") as archive:\n        archive.add(source, arcname=source.name)\n    with tarfile.open(temporary, \"r\") as archive:\n        members = {member.name: member for member in archive.getmembers()}\n        for name in required:\n            member = members[f\"{source.name}/{name}\"]\n            assert member.size == (source / name).stat().st_size\n    # A partial archive never replaces the previous completed archive.\n    os.replace(temporary, destination)\n    return {\"path\": str(destination), \"bytes\": destination.stat().st_size}\n\n\ndef copy_predictions(sql_path, records_path, destination):\n    destination.mkdir(parents=True, exist_ok=True)\n    shutil.copy2(sql_path, destination / \"dev.sql\")\n    shutil.copy2(records_path, destination / \"dev.pkl\")\n\n\ndef run(backup):\n    import torch\n    from load_data import get_dataloader\n    from t5_utils import (\n        initialize_model, initialize_optimizer_and_scheduler,\n        save_model, load_model_from_checkpoint,\n    )\n    from train_t5 import train_epoch, eval_epoch\n    from utils import set_random_seeds\n\n    backup = Path(backup).resolve()\n    settings = json.loads((backup / \"configuration.json\").read_text())\n    args = SimpleNamespace(**settings)\n    assert args.finetune and not args.use_wandb\n    assert args.max_n_epochs > args.num_warmup_epochs >= 0\n    assert args.patience_epochs > 0 and args.learning_rate > 0\n    assert torch.cuda.is_available(), \"Connect to a GPU before starting.\"\n    set_random_seeds(args.seed)\n    started = time.monotonic()\n    status = {\"state\": \"RUNNING\", \"stage\": \"initializing\", \"completed_epochs\": 0,\n              \"backup_folder\": str(backup), \"held_out_test_used\": False}\n\n    def update(stage, **extra):\n        status.update(stage=stage, updated_utc=datetime.now(timezone.utc).isoformat(), **extra)\n        write_json(backup / \"status.json\", status)\n\n    try:\n        update(\"loading_train_and_dev\")\n        # Deliberately avoid load_t5_data()/main(): no test dataset is loaded.\n        train_loader = get_dataloader(args.batch_size, \"train\")\n        dev_loader = get_dataloader(args.test_batch_size, \"dev\")\n        assert len(train_loader.dataset) == 4225 and len(dev_loader.dataset) == 466\n        model = initialize_model(args)\n        optimizer, scheduler = initialize_optimizer_and_scheduler(args, model, len(train_loader))\n        checkpoint_dir = Path(\"checkpoints/ft_experiments\") / args.experiment_name\n        Path(\"results\").mkdir(exist_ok=True)\n        Path(\"records\").mkdir(exist_ok=True)\n        sql_path, records_path = \"results/current_dev.sql\", \"records/current_dev.pkl\"\n        gt_sql, gt_records = \"data/dev.sql\", \"records/ground_truth_dev.pkl\"\n        assert Path(gt_records).is_file()\n        metadata = {\n            \"settings\": settings, \"gpu\": torch.cuda.get_device_name(0),\n            \"parameters\": sum(p.numel() for p in model.parameters()),\n            \"trainable_parameters\": sum(p.numel() for p in model.parameters() if p.requires_grad),\n            \"train_examples\": len(train_loader.dataset), \"dev_examples\": len(dev_loader.dataset),\n            \"train_batches_per_epoch\": len(train_loader),\n            \"maximum_optimizer_steps\": len(train_loader) * args.max_n_epochs,\n            \"warmup_steps\": len(train_loader) * args.num_warmup_epochs,\n            \"precision\": \"float32\", \"gradient_accumulation_steps\": 1,\n            \"gradient_clipping\": None, \"checkpoint_selection\": \"highest dev record F1; first on ties\",\n            \"pretrained_checkpoint\": \"google-t5/t5-small\",\n            \"checkpoint_contents\": \"weights/config/tokenizer, NOT optimizer/scheduler/RNG state\",\n            \"versions\": {name: importlib.metadata.version(name) for name in\n                         [\"torch\", \"transformers\", \"tokenizers\", \"numpy\"]},\n        }\n        write_json(backup / \"run_metadata.json\", metadata)\n        print(json.dumps(metadata, indent=2), flush=True)\n        print(\"FULL TRAINING STARTING. Only train/dev; held-out test remains untouched.\", flush=True)\n        history, best_f1, best_epoch, stale = [], -1.0, None, 0\n        for epoch in range(1, args.max_n_epochs + 1):\n            epoch_start = time.monotonic()\n            update(\"training\", current_epoch=epoch)\n            print(f\"\\nEPOCH {epoch}/{args.max_n_epochs}: training {len(train_loader)} batches\", flush=True)\n            train_loss = float(train_epoch(args, model, train_loader, optimizer, scheduler))\n            if not math.isfinite(train_loss):\n                raise ValueError(\"Non-finite training loss; stop and inspect.\")\n\n            # Protect the completed epoch BEFORE potentially slower dev generation.\n            update(\"backing_up_last_checkpoint\", current_epoch=epoch)\n            save_model(str(checkpoint_dir), model, best=False)\n            write_json(checkpoint_dir / \"last/checkpoint_meta.json\", {\n                \"epoch\": epoch, \"train_loss\": train_loss, \"dev_evaluation_completed\": False,\n                \"configuration\": settings,\n            })\n            last_info = publish_checkpoint(checkpoint_dir / \"last\", backup / \"checkpoints/last.tar\")\n            update(\"evaluating_dev\", completed_epochs=epoch, last_checkpoint=last_info)\n            print(f\"EPOCH {epoch}: TRAINED WEIGHTS BACKED UP TO DRIVE before dev evaluation.\", flush=True)\n\n            values = eval_epoch(args, model, dev_loader, gt_sql, sql_path, gt_records, records_path)\n            loss, f1, record_em, sql_em, error_rate = map(float, values)\n            if not all(math.isfinite(v) for v in [loss, f1, record_em, sql_em, error_rate]):\n                raise ValueError(\"Non-finite dev metrics; stop and inspect.\")\n            row = dict(epoch=epoch, train_loss=train_loss, dev_loss=loss,\n                       record_f1=f1, record_em=record_em, sql_em=sql_em, sql_error_rate=error_rate)\n            copy_predictions(sql_path, records_path, backup / \"predictions/last\")\n            improved = f1 > best_f1\n            if improved:\n                best_f1, best_epoch, stale = f1, epoch, 0\n                update(\"backing_up_best_checkpoint\")\n                save_model(str(checkpoint_dir), model, best=True)\n                write_json(checkpoint_dir / \"best/checkpoint_meta.json\", {\n                    **row, \"dev_evaluation_completed\": True, \"configuration\": settings,\n                })\n                best_info = publish_checkpoint(checkpoint_dir / \"best\", backup / \"checkpoints/best.tar\")\n                copy_predictions(sql_path, records_path, backup / \"predictions/best\")\n                status[\"best_checkpoint\"] = best_info\n            else:\n                stale += 1\n            row.update(seconds=time.monotonic() - epoch_start, improved=improved)\n            history.append(row)\n            write_json(backup / \"history.json\", history)\n            write_json(backup / \"best_metrics.json\", next(r for r in history if r[\"epoch\"] == best_epoch))\n            update(\"epoch_completed\", best_epoch=best_epoch, best_dev_record_f1=best_f1,\n                   epochs_without_improvement=stale)\n            print(f\"EPOCH {epoch} COMPLETE: train loss={train_loss:.4f}, dev loss={loss:.4f}, \"\n                  f\"Record F1={100*f1:.2f}%, Record EM={100*record_em:.2f}%, \"\n                  f\"SQL EM={100*sql_em:.2f}%, SQL errors={100*error_rate:.2f}%\", flush=True)\n            print(f\"Epoch wall time including dev/backups: {row['seconds']/60:.1f} minutes. \"\n                  f\"Best epoch={best_epoch}; saved to Drive.\", flush=True)\n            if stale >= args.patience_epochs:\n                print(f\"Early stopping: {stale} epochs without improved dev Record F1.\", flush=True)\n                break\n\n        update(\"reloading_best_for_final_dev_check\")\n        del optimizer, scheduler, model\n        gc.collect()\n        torch.cuda.empty_cache()\n        best_model = load_model_from_checkpoint(args, best=True)\n        values = eval_epoch(args, best_model, dev_loader, gt_sql, \"results/best_dev.sql\",\n                            gt_records, \"records/best_dev.pkl\")\n        loss, f1, record_em, sql_em, error_rate = map(float, values)\n        assert math.isclose(f1, best_f1, rel_tol=1e-6, abs_tol=1e-8), \"Reloaded best dev F1 differs.\"\n        copy_predictions(\"results/best_dev.sql\", \"records/best_dev.pkl\", backup / \"predictions/best\")\n        final = {\"state\": \"COMPLETED\", \"experiment_name\": args.experiment_name,\n                 \"epochs_run\": len(history), \"best_epoch\": best_epoch,\n                 \"dev_loss\": loss, \"record_f1\": f1, \"record_em\": record_em,\n                 \"sql_em\": sql_em, \"sql_error_rate\": error_rate,\n                 \"seconds\": time.monotonic() - started, \"held_out_test_used\": False,\n                 \"best_checkpoint_archive\": str(backup / \"checkpoints/best.tar\"),\n                 \"backup_folder\": str(backup)}\n        write_json(backup / \"final_report.json\", final)\n        update(\"completed\", state=\"COMPLETED\", final_report=final)\n        print(\"\\nPART 2 TRAINING AND DEV BACKUP COMPLETE\", flush=True)\n        print(json.dumps(final, indent=2), flush=True)\n        print(\"Save this notebook and ask for review. Do not submit diagnostic outputs or generate test yet.\")\n    except BaseException as error:\n        import traceback\n        update(\"failed_or_interrupted\", state=\"FAILED_OR_INTERRUPTED\", error=traceback.format_exc())\n        print(\"Run did not complete. Existing completed Drive archives/logs remain available:\", backup, flush=True)\n        raise\n\n\nif __name__ == \"__main__\":\n    run(sys.argv[1])\n"

# Operational notebook preparation. The notebook embeds RUNNER_SOURCE above this.
from pathlib import Path
from datetime import datetime, timezone
from google.colab import drive
import hashlib
import json
import shutil
import subprocess

PART2 = Path("/content/part2")
PYTHON2 = "/content/hw4-part2-env/bin/python"
assert Path(PYTHON2).is_file(), "Use the same Part 2 Colab runtime/environment."
for name in ["load_data.py", "t5_utils.py", "train_t5.py", "utils.py", "requirements.txt"]:
    assert (PART2 / name).is_file(), f"Upload part2 first: missing {name}"
subprocess.run([PYTHON2, "-c", "import torch; assert torch.cuda.is_available(), 'Connect to a GPU'; print('GPU:', torch.cuda.get_device_name(0))"], check=True)
assert shutil.disk_usage("/content").free > 2 * 1024**3, "Need at least 2 GiB free local disk."
drive.mount("/content/drive")
base = Path("/content/drive/MyDrive/NYU_NLP_HW4/part2")

# Require the exact source files that passed the evaluation check.
passed = []
for candidate in sorted((base / "q5_eval_checks").glob("*"), reverse=True):
    status_file = candidate / "status.json"
    if status_file.is_file() and json.loads(status_file.read_text()).get("state") == "COMPLETED":
        passed.append(candidate)
assert passed, "First finish Section 9 in HW4_Part2_Colab.ipynb."
checked = passed[0]
for name in ["load_data.py", "t5_utils.py", "train_t5.py", "utils.py"]:
    assert (PART2 / name).read_bytes() == (checked / name).read_bytes(), (
        f"{name} differs from the source that passed Section 9. Re-run that check first."
    )

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
T5_WORK = Path("/content") / f"hw4_t5_train_{stamp}"
T5_BACKUP = base / "training_runs" / stamp
T5_WORK.mkdir(exist_ok=False)
T5_BACKUP.mkdir(parents=True, exist_ok=False)
source_backup = T5_BACKUP / "source"
source_backup.mkdir()
for source in PART2.glob("*.py"):
    shutil.copy2(source, T5_WORK / source.name)
    shutil.copy2(source, source_backup / source.name)
shutil.copy2(PART2 / "requirements.txt", source_backup / "requirements.txt")
shutil.copytree(PART2 / "data", T5_WORK / "data")
shutil.copytree(PART2 / "data", source_backup / "data")
(T5_WORK / "records").mkdir()
shutil.copy2(PART2 / "records/ground_truth_dev.pkl", T5_WORK / "records/ground_truth_dev.pkl")
shutil.copy2(PART2 / "records/ground_truth_dev.pkl", source_backup / "ground_truth_dev.pkl")
for name in ["environment.json", "pip_freeze.txt"]:
    artifact = Path("/content/hw4-part2-artifacts") / name
    if artifact.is_file():
        shutil.copy2(artifact, T5_BACKUP / name)

T5_SETTINGS = {
    "finetune": True, "experiment_name": "experiment", "seed": 42,
    "optimizer_type": "AdamW", "learning_rate": 3e-4, "weight_decay": 0.0,
    "scheduler_type": "linear", "num_warmup_epochs": 1,
    "max_n_epochs": 20, "patience_epochs": 4,
    "batch_size": 8, "test_batch_size": 8, "use_wandb": False,
    "max_new_tokens": 512, "num_beams": 1,
}
(T5_BACKUP / "configuration.json").write_text(json.dumps(T5_SETTINGS, indent=2))
manifest = {
    str(p.relative_to(source_backup)): hashlib.sha256(p.read_bytes()).hexdigest()
    for p in source_backup.rglob("*") if p.is_file()
}
(T5_BACKUP / "source_sha256.json").write_text(json.dumps(manifest, indent=2))
(T5_BACKUP / "passed_evaluation_check.txt").write_text(str(checked))
T5_RUNNER = T5_WORK / "run_with_drive_backup.py"
T5_RUNNER.write_text(RUNNER_SOURCE)
shutil.copy2(T5_RUNNER, T5_BACKUP / T5_RUNNER.name)
(T5_BACKUP / "status.json").write_text(json.dumps({
    "state": "PREPARED_NOT_STARTED", "backup_folder": str(T5_BACKUP),
    "working_folder": str(T5_WORK), "configuration": T5_SETTINGS,
}, indent=2))
print("Preparation complete. TRAINING HAS NOT STARTED.")
print("Working folder:", T5_WORK)
print("SAVE THIS DRIVE LOCATION:", T5_BACKUP)
print(json.dumps(T5_SETTINGS, indent=2))
print("Run the next cell ONCE to start full training. Keep at least ~1 GB available in Drive.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Preparation complete. TRAINING HAS NOT STARTED.
Working folder: /content/hw4_t5_train_20261110T032045677993Z
SAVE THIS DRIVE LOCATION: /content/drive/MyDrive/NYU_NLP_HW4/part2/training_runs/20261110T032045677993Z
{
  "finetune": true,
  "experiment_name": "experiment",
  "seed": 42,
  "optimizer_type": "AdamW",
  "learning_rate": 0.0003,
  "weight_decay": 0.0,
  "scheduler_type": "linear",
  "num_warmup_epochs": 1,
  "max_n_epochs": 20,
  "patience_epochs": 4,
  "batch_size": 8,
  "test_batch_size": 8,
  "use_wandb": false,
  "max_new_tokens": 512,
  "num_beams": 1
}
Run the next cell ONCE to start full training. Keep at least ~1 GB available in Drive.


## 2. Train — run once

This starts real training, up to 529 training batches per epoch. Epoch summaries include dev metrics and elapsed time. Wait for `EPOCH 1: TRAINED WEIGHTS BACKED UP TO DRIVE` before relying on a trained-model backup.


In [2]:
# Run once. This is the long-running cell that actually trains the model.
import json
import subprocess
from datetime import datetime, timezone

assert "T5_WORK" in globals() and T5_RUNNER.is_file(), "Run preparation first."
assert json.loads((T5_BACKUP / "status.json").read_text())["state"] == "PREPARED_NOT_STARTED", (
    "This run was already started. Do not accidentally train twice."
)
with (T5_WORK / ".training_started").open("x") as marker:
    marker.write(datetime.now(timezone.utc).isoformat())

print("STARTING FULL TRAINING. Drive backup:", T5_BACKUP, flush=True)
print("The first Drive model checkpoint appears after the first training epoch.", flush=True)
process = None
try:
    with (T5_BACKUP / "training.log").open("w", buffering=1) as log:
        process = subprocess.Popen(
            [PYTHON2, "-u", str(T5_RUNNER), str(T5_BACKUP)],
            cwd=T5_WORK, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1,
        )
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
            log.flush()
        return_code = process.wait()
        if return_code:
            raise RuntimeError(f"Training process exited with code {return_code}. Inspect training.log.")
except BaseException:
    if process is not None and process.poll() is None:
        process.terminate()
        try:
            process.wait(timeout=15)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait(timeout=15)
    status_path = T5_BACKUP / "status.json"
    status = json.loads(status_path.read_text()) if status_path.is_file() else {}
    status.update(state="FAILED_OR_INTERRUPTED", updated_utc=datetime.now(timezone.utc).isoformat())
    status_path.write_text(json.dumps(status, indent=2))
    print("Stopped or failed. Completed Drive checkpoints/logs remain at:", T5_BACKUP)
    raise
else:
    assert json.loads((T5_BACKUP / "status.json").read_text())["state"] == "COMPLETED"
    print("\nTRAINING COMPLETE. Everything needed for review is in:", T5_BACKUP)
    print("Save this notebook. Do not rerun training; ask for review of the dev results.")


STARTING FULL TRAINING. Drive backup: /content/drive/MyDrive/NYU_NLP_HW4/part2/training_runs/20261110T032045677993Z
The first Drive model checkpoint appears after the first training epoch.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
/content/hw4-part2-env/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
{
  "settings": {
    "finetune": true,
    "experiment_name": "experiment",
    "seed": 42,
    "optimizer_type": "AdamW",
    "learning_rate": 0.0003,
    "weight_decay": 0.0,
    "scheduler_type": "linear",
    "num_warmup_epochs": 1,
    "max_n_epochs": 20,
    "patience_epochs": 4,
    "batch_size": 8,
    "test_batch_size": 8,
    "use_wandb": false,
    "max_new_tokens": 512,
    "num_beams": 1


: 

## 3. Inspect saved progress — read-only

Run after completion or after reconnecting. If notebook variables are gone, this reads the newest training-run folder from Drive. It does not restart training.


In [1]:
# Read-only: safe to run after reconnecting, without re-running training.
from pathlib import Path
from google.colab import drive
import json

drive.mount("/content/drive")
if "T5_BACKUP" not in globals():
    runs = sorted(Path("/content/drive/MyDrive/NYU_NLP_HW4/part2/training_runs").glob("*"))
    assert runs, "No saved Part 2 training runs found."
    T5_BACKUP = runs[-1]
print("Inspecting:", T5_BACKUP)
for name in ["status.json", "best_metrics.json", "final_report.json"]:
    path = T5_BACKUP / name
    if path.is_file():
        print("\n" + name + ":\n" + path.read_text())
for name in ["best.tar", "last.tar"]:
    path = T5_BACKUP / "checkpoints" / name
    print(name, f"{path.stat().st_size / 1024**2:.1f} MiB" if path.is_file() else "not yet saved")
log = T5_BACKUP / "training.log"
if log.is_file():
    print("\nLast log lines:\n" + "\n".join(log.read_text(errors="replace").splitlines()[-15:]))
print("A stale RUNNING status after runtime loss does not prove the process is still alive.")


Mounted at /content/drive
Inspecting: /content/drive/MyDrive/NYU_NLP_HW4/part2/training_runs/20261110T032045677993Z

status.json:
{
  "state": "RUNNING",
  "stage": "evaluating_dev",
  "completed_epochs": 4,
  "backup_folder": "/content/drive/MyDrive/NYU_NLP_HW4/part2/training_runs/20261110T032045677993Z",
  "held_out_test_used": false,
  "updated_utc": "2026-11-10T03:53:30.031630+00:00",
  "current_epoch": 4,
  "last_checkpoint": {
    "path": "/content/drive/MyDrive/NYU_NLP_HW4/part2/training_runs/20261110T032045677993Z/checkpoints/last.tar",
    "bytes": 245299200
  },
  "best_checkpoint": {
    "path": "/content/drive/MyDrive/NYU_NLP_HW4/part2/training_runs/20261110T032045677993Z/checkpoints/best.tar",
    "bytes": 245299200
  },
  "best_epoch": 3,
  "best_dev_record_f1": 0.494474250584802,
  "epochs_without_improvement": 0
}

best_metrics.json:
{
  "epoch": 3,
  "train_loss": 0.12929363247530626,
  "dev_loss": 0.0725661679995442,
  "record_f1": 0.494474250584802,
  "record_em": 0.